In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))
sys.path.insert(0, os.path.abspath("../sdlarch-rl"))


from sdlarch_rl.utils.utils import get_last_index, GenericCNN, TimeLimit, FrameSkip
from sdlarch_rl import make
from stable_baselines3.common.atari_wrappers import WarpFrame
import time
import cv2
import numpy as np
import keyboard
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
from pathlib import Path
import pygame
from inputs import get_gamepad, devices
import threading
from imitation.data.types import Trajectory
import torch as th
import os
import re
from utils import get_last_index, LSTMWrapper
import gymnasium as gym
from final_fight import FinalFightActionWrapper, make_env
from stable_baselines3.common.policies import ActorCriticPolicy

env = make_vec_env(make_env(), n_envs=1, vec_env_cls=DummyVecEnv)

# --- CONFIGS ---
MAX_TRAJ = 10
demo_path = 'demos/'
SCALE = 1/4
SCREEN_WIDTH = 854 # int(640 * SCALE)
SCREEN_HEIGHT = 480 # int(480 * SCALE)
NUMBER_OF_ACTIONS = 6
MAX_FPS=30
IS_HG_DAGGER = True
os.makedirs(demo_path, exist_ok=True)

policy = None

if IS_HG_DAGGER:
    train_path = "./imitation/"
    sub_train_path = train_path + "steps"
    last_index_imitation = get_last_index(str(sub_train_path), "bc_policy", "zip")
    current_epoch = last_index_imitation
    
    train_path="./imitation/"
    sub_train_path = train_path + "steps"

    # todo: comment below to change current_epoch to smart value
    current_epoch = 30
    latest_model_path = sub_train_path + f"/bc_policy{current_epoch}.zip"
    policy = ActorCriticPolicy.load(latest_model_path)
    
    policy = LSTMWrapper(policy)
    policy.reset()


class RendererThread(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.img = None
        self.color = (0, 0, 255)
        self.count_record = 0
        self.running = True
        self.lock = threading.Lock()
        self.fps = 0
        self.use_ai = False
        self.current_epoch = 0

    def update_data(self, img, color, count, fps, use_ai, current_epoch):
        with self.lock:
            self.img = img
            self.color = color
            self.count_record = count
            self.fps = fps
            self.use_ai = use_ai
            self.current_epoch = current_epoch

    def run(self):
        pygame.init()
        window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.HWSURFACE | pygame.DOUBLEBUF)
        pygame.display.set_caption("Final Fight Capture")
        font = pygame.font.SysFont("Arial", 24)
        
        while self.running:
            pygame.event.pump()
            
            img_to_render = None
            with self.lock:
                if self.img is not None:
                    img_to_render = self.img
                    current_color = self.color
                    current_count = self.count_record
                    current_fps = self.fps
                    use_ai = self.use_ai
                    current_epoch = self.current_epoch

            if img_to_render is not None:
                img_rgb = cv2.cvtColor(img_to_render, cv2.COLOR_BGR2RGB)
                surface = pygame.image.frombuffer(img_rgb.flatten(), (SCREEN_WIDTH, SCREEN_HEIGHT), 'RGB')
                
                window.blit(surface, (0, 0))


                current_epoch_text = font.render(f"Current Epoch: {int(current_epoch)}", True, (255, 255, 255))
                fps_text = font.render(f"FPS: {int(current_fps)}", True, (255, 255, 255))
                count_text = font.render(f"Demos: {current_count}/{MAX_TRAJ}", True, (255, 255, 255))
                status_color = (0, 255, 0) if current_color == (0, 255, 0) else (255, 0, 0)
                rec_text = font.render("RECORDING" if current_color == (0, 255, 0) else "IDLE", True, status_color)

                window.blit(current_epoch_text, (10, 10))
                window.blit(fps_text, (10, 40))
                window.blit(count_text, (10, 70))
                window.blit(rec_text, (10, 100))

                if use_ai:
                    use_ai_text = font.render("USING AI", True, (0, 255, 0))
                    window.blit(use_ai_text, (10, 130))

                
                pygame.display.flip()
            
            time.sleep(1/30)


last_index = -1
for p in Path(demo_path).iterdir():
    m = re.search(r"demos(\d+)\.pt$", p.name)
    if m: last_index = max(last_index, int(m.group(1)))

gamepad = devices.gamepads[0]
STATE = {k: 0 for k in ["UP", "DOWN", "LEFT", "RIGHT", "A", "B", "X", "START", "CAM_X", "CAM_Y", "LT", "RT", "L3"]}
lock = threading.Lock()

DEADZONE = 10000
NORM = 32768

def input_loop():
    while True:
        for e in gamepad.read():
            with lock:
                if e.code == "ABS_HAT0X":
                    STATE["LEFT"]  = 1 if e.state == -1 else 0
                    STATE["RIGHT"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_HAT0Y":
                    STATE["UP"]   = 1 if e.state == -1 else 0
                    STATE["DOWN"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_X":
                    if e.state > DEADZONE:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 1
                    elif e.state < -DEADZONE:
                        STATE["LEFT"] = 1
                        STATE["RIGHT"] = 0
                    else:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 0

                elif e.code == "ABS_Y":
                    if e.state > DEADZONE:
                        STATE["UP"] = 1
                        STATE["DOWN"] = 0
                    elif e.state < -DEADZONE:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 1
                    else:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 0

                # RIGHT STICK X
                elif e.code == "ABS_RX":
                    if abs(e.state) > DEADZONE:
                        STATE["CAM_X"] = e.state / NORM
                    else:
                        STATE["CAM_X"] = 0

                # RIGHT STICK Y
                elif e.code == "ABS_RY":
                    if abs(e.state) > DEADZONE:
                        STATE["CAM_Y"] = e.state / NORM
                    else:
                        STATE["CAM_Y"] = 0

                elif e.code == "BTN_SOUTH":
                    STATE["A"] = e.state

                elif e.code == "BTN_EAST":
                    STATE["B"] = e.state

                elif e.code == "BTN_WEST":
                    STATE["X"] = e.state

                elif e.code == "BTN_START" or e.code == "BTN_MENU" \
                    or e.code == "BTN_SELECT" or e.code == "BTN_MODE"  \
                    or e.code == "BTN_OPTIONS":
                    STATE["START"] = e.state

                elif e.code == "BTN_THUMBL":
                    STATE["L3"] = e.state

                elif e.code == "ABS_Z":
                    value = e.state / 255.0
                    if value < 0.5:
                        value = 0
                    STATE["LT"] = value
                
                elif e.code == "ABS_RZ":
                    value = e.state / 255.0
                    if value < 0.5:
                        value = 0
                    STATE["RT"] = value

DEAD_NORM = DEADZONE / NORM

def discretize(value):

    # neutral
    if abs(value) <= DEAD_NORM:
        return 0

    # positive
    if value > 0:
        if value < DEAD_NORM + 0.5:
            return 1
        else:
            return 2

    # negative
    else:
        if value > -(DEAD_NORM + 0.5):
            return 3
        else:
            return 4

t_input = threading.Thread(target=input_loop, daemon=True)
t_input.start()

# Start render thread
renderer = RendererThread()
renderer.start()

is_running = False
is_recording = False
count_record = 0
trajectories = []
recorded_obs = []
recorded_actions = []

print("Done! Press 'K' to start/stop the record.")

fps_avg_frame_count = 0
fps_start_time = time.time()
actual_fps = 0

obs = env.reset()
is_paused = False

ram = env.envs[0].unwrapped.em.get_ram()
print(hex(ram.ctypes.data))
use_ai = True
pred_act = None

while count_record < MAX_TRAJ:
    loop_start = time.time()
    
    if keyboard.is_pressed('k'):
        is_running = not is_running
        time.sleep(0.3)
        print(f"Recoding: {is_running}")

    action = [np.zeros(NUMBER_OF_ACTIONS, dtype=np.float32)]

    with lock:
        received_action = [
            STATE["UP"], # 0 
            STATE["DOWN"], 
            STATE["LEFT"], 
            STATE["RIGHT"],
            STATE["A"], 
            STATE["B"], 
            STATE["X"],
            STATE["LT"], # 7
            STATE["RT"],
            STATE["L3"],
            STATE["CAM_X"], # 10
            STATE["CAM_Y"], # 11
            STATE["START"],
        ]

        if keyboard.is_pressed('up') or received_action[0]: action[0][0] = 1
        if keyboard.is_pressed('down') or received_action[1]: action[0][1] = 1
        if keyboard.is_pressed('left') or received_action[2]: action[0][2] = 1
        if keyboard.is_pressed('right') or received_action[3]: action[0][3] = 1
        if keyboard.is_pressed('i') or received_action[4]: action[0][4] = 1
        if received_action[6]: action[0][5] = 1

        if keyboard.is_pressed('r'):
            env.reset()
            use_ai = True
            time.sleep(0.3)

        if keyboard.is_pressed('p'):
            is_paused = not is_paused
            time.sleep(0.3)

        if received_action[12]:
            use_ai = not use_ai
            time.sleep(0.15)
            

    if is_paused:
        continue

    if IS_HG_DAGGER:
        pred_act, _ = policy.predict(obs, deterministic=False)

    if use_ai and IS_HG_DAGGER and is_running:
        action = pred_act

    obs, rew, done, info = env.step(action)

    # print(info)
    # print(rew)
    # print(done[0])

    
    img = env.render()
    if obs is not None:
        # img = obs[0, :, :, -1]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized =  cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT), interpolation=cv2.INTER_NEAREST)
        color = (0, 255, 0) if is_running else (0, 0, 255)
        renderer.update_data(img_resized, color, count_record, actual_fps, use_ai, current_epoch)

    if is_running:
        if not is_recording: # first OBS
            env.reset()
            use_ai = True
        recorded_obs.append(obs)
        recorded_actions.append(action[0])
        is_recording = True
    elif is_recording:
        print(f"Finish trajectory {count_record}")
        obs_uint8 = np.stack([o.astype(np.uint8) for o in recorded_obs], axis=0)
        print("Obs shape: ", obs_uint8.shape)
        del recorded_actions[-1]
        trajectories.append(Trajectory(obs=obs_uint8, acts=np.array(recorded_actions), infos=None, terminal=False))
        recorded_obs, recorded_actions = [], []
        count_record += 1
        is_recording = False
        env.reset()
        use_ai = True
        
        if done[0]:
            env.reset()
            is_running = False
            is_recording = True
            use_ai = True


    fps_avg_frame_count += 1
    if time.time() - fps_start_time >= 1.0: # update fps every 1 second
        actual_fps = fps_avg_frame_count / (time.time() - fps_start_time)
        fps_avg_frame_count = 0
        fps_start_time = time.time()
        # print(f"FPS of capture: {actual_fps:.2f}")

    # try fix fps (25)
    time_to_wait = (1/40) - (time.time() - loop_start)
    if time_to_wait > 0:
        time.sleep(time_to_wait)

renderer.running = False
print("Saving to file...")
th.save(trajectories, os.path.join(demo_path, f"demos{last_index + 1}.pt"))
print("Save completed...")
pygame.quit()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_variables = th.load(path, map_location=device)


Done! Press 'K' to start/stop the record.
0x1d9009801b0
Recoding: True
Recoding: False
Finish trajectory 0
Obs shape:  (2, 1, 96, 96, 1)
